# EDA - Explanatory Data Analysis
- Разведочного анализа данных

1. Импортирование библиотек

In [12]:
import pandas as pd
import os
import random
import re
import sys
import csv
import yaml

Конфигурация

In [13]:
with open('D://HereAgain//CALOIRO 2025//Programming//LAB_MAI_01//Config//paramts.yaml', 'r') as config_file:
    config = yaml.safe_load(config_file)

os.chdir(config['base']['root_project_dir'])

print(config)

{'base': {'random_state': 42, 'test_size': 0.2, 'root_project_dir': 'D:\\HereAgain\\CALOIRO 2025\\Programming\\LAB_MAI_01'}, 'data': {'raw_dataset_csv': 'Data/test_scores.csv'}, 'reports': {'ydata_report_path': 'reports/ydata_report1.html', 'dvclive_path': 'dvclive/', 'metrics_path': 'reports/metrics/', 'figures_path': 'reports/figures/'}}


2. Загрузка данных

In [14]:
df = pd.read_csv(config['data']['raw_dataset_csv'])
df

,school,school_setting,school_type,classroom,teaching_method,n_student,student_id,gender,lunch,pretest,posttest
0,ANKYI,Urban,Non-public,6OL,Standard,20.0,2FHT3,Female,Does not qualify,62.0,72.0
1,ANKYI,Urban,Non-public,6OL,Standard,20.0,3JIVH,Female,Does not qualify,66.0,79.0
2,ANKYI,Urban,Non-public,6OL,Standard,20.0,3XOWE,Male,Does not qualify,64.0,76.0
3,ANKYI,Urban,Non-public,6OL,Standard,20.0,556O0,Female,Does not qualify,61.0,77.0
4,ANKYI,Urban,Non-public,6OL,Standard,20.0,74LOE,Male,Does not qualify,64.0,76.0
...,...,...,...,...,...,...,...,...,...,...,...
1979,ZOWMK,Urban,Public,ZBH,Standard,30.0,S4I5S,Male,Qualifies for reduced/free lunch,39.0,50.0
1980,ZOWMK,Urban,Public,ZBH,Standard,30.0,T8LSK,Female,Does not qualify,39.0,55.0
1981,ZOWMK,Urban,Public,ZBH,Standard,30.0,VNP26,Female,Qualifies for reduced/free lunch,38.0,46.0
1982,ZOWMK,Urban,Public,ZBH,Standard,30.0,YUEIH,Male,Qualifies for reduced/free lunch,46.0,53.0


In [15]:
df.describe()

,n_student,pretest,posttest
count,1984.000000,1984.000000,1984.00000
mean,22.769153,54.980847,67.12500
std,4.248477,13.558027,13.93532
min,14.000000,22.000000,32.00000
25%,20.000000,44.000000,56.00000
50%,22.000000,56.000000,68.00000
75%,27.000000,65.000000,77.00000
max,31.000000,93.000000,100.00000


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1984 entries, 0 to 1983
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   school           1984 non-null   object 
 1   school_setting   1984 non-null   object 
 2   school_type      1984 non-null   object 
 3   classroom        1984 non-null   object 
 4   teaching_method  1984 non-null   object 
 5   n_student        1984 non-null   float64
 6   student_id       1984 non-null   object 
 7   gender           1984 non-null   object 
 8   lunch            1984 non-null   object 
 9   pretest          1984 non-null   float64
 10  posttest         1984 non-null   float64
dtypes: float64(3), object(8)
memory usage: 170.6+ KB


In [17]:
df.var(numeric_only=True)

n_student     18.049557
pretest      183.820107
posttest     194.193142
dtype: float64

Проверка наличия пустых строк 

In [18]:
df.isnull().sum()

school             0
school_setting     0
school_type        0
classroom          0
teaching_method    0
n_student          0
student_id         0
gender             0
lunch              0
pretest            0
posttest           0
dtype: int64

В данных отсутствуют пустые строки

3. Подготовка признаков

3.1. Удаление не полезных признаков

1) Характеристика "student_id" содержит множество уникальных категорий, и они просто являются идентификаторами, не несут важный смыл для обучения.

2) У характеристик "school" и "classroom" также имеют слишком много категорий, что преоброзует много колонок при кодировании с помощью "OneHotEncoder", и также имеет риск переобучения.

3) Поэтому мы будем пренебрегать эти характеристики/признаки: student_id, school и classroom.

In [19]:
df.drop(['student_id', 'school', 'classroom'], axis=1, inplace=True)
df

,school_setting,school_type,teaching_method,n_student,gender,lunch,pretest,posttest
0,Urban,Non-public,Standard,20.0,Female,Does not qualify,62.0,72.0
1,Urban,Non-public,Standard,20.0,Female,Does not qualify,66.0,79.0
2,Urban,Non-public,Standard,20.0,Male,Does not qualify,64.0,76.0
3,Urban,Non-public,Standard,20.0,Female,Does not qualify,61.0,77.0
4,Urban,Non-public,Standard,20.0,Male,Does not qualify,64.0,76.0
...,...,...,...,...,...,...,...,...
1979,Urban,Public,Standard,30.0,Male,Qualifies for reduced/free lunch,39.0,50.0
1980,Urban,Public,Standard,30.0,Female,Does not qualify,39.0,55.0
1981,Urban,Public,Standard,30.0,Female,Qualifies for reduced/free lunch,38.0,46.0
1982,Urban,Public,Standard,30.0,Male,Qualifies for reduced/free lunch,46.0,53.0


3.2.  Подготовка категориальных признаков

In [20]:
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer

for col in ['gender', 'lunch', 'school_type', 'teaching_method']:
    lb = LabelBinarizer()
    df[col] = lb.fit_transform(df[col])

for col in ['school_setting']:
    ohe = OneHotEncoder()
    encoded = ohe.fit_transform(df[[col]]).toarray()
    df = df.join(pd.DataFrame(encoded, columns=[f"{col}_{c}" for c in ohe.categories_[0]], index=df.index))
    df.drop(col, axis=1, inplace=True)   
df

,school_type,teaching_method,n_student,gender,lunch,pretest,posttest,school_setting_Rural,school_setting_Suburban,school_setting_Urban
0,0,1,20.0,0,0,62.0,72.0,0.0,0.0,1.0
1,0,1,20.0,0,0,66.0,79.0,0.0,0.0,1.0
2,0,1,20.0,1,0,64.0,76.0,0.0,0.0,1.0
3,0,1,20.0,0,0,61.0,77.0,0.0,0.0,1.0
4,0,1,20.0,1,0,64.0,76.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
1979,1,1,30.0,1,1,39.0,50.0,0.0,0.0,1.0
1980,1,1,30.0,0,0,39.0,55.0,0.0,0.0,1.0
1981,1,1,30.0,0,1,38.0,46.0,0.0,0.0,1.0
1982,1,1,30.0,1,1,46.0,53.0,0.0,0.0,1.0


Для кодирования признаков, содержащих только две категории или значения (бинарные признаки, например, пол) используется "LabelBinarizer". А для кодирования признаков, содержащих больше, чем две категории применяется "OneHotEncoder".
1) gender:
    1 - male
    0 - female

2) lunch:
    1 - Qualifies to reduced/free lunch
    0 - Does not qualify

3) school type:
    1 - Public
    0 - Non-public

4) teaching method:
    1 - Standard
    0 - Experimental


3.3. Подготовка числовых признаков

1) Для числовых признаков (n_student и pretest) выполняем НОРМАЛИЗАЦИЮ, так как у них высокие дисперсии.
2) protest - это целевая функция (target),то есть, что хотим предсказать, поэтому не трогаем.

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[['n_student','pretest']] = scaler.fit_transform(df[['n_student','pretest']])
df

,school_type,teaching_method,n_student,gender,lunch,pretest,posttest,school_setting_Rural,school_setting_Suburban,school_setting_Urban
0,0,1,-0.651963,0,0,0.517843,72.0,0.0,0.0,1.0
1,0,1,-0.651963,0,0,0.812945,79.0,0.0,0.0,1.0
2,0,1,-0.651963,1,0,0.665394,76.0,0.0,0.0,1.0
3,0,1,-0.651963,0,0,0.444067,77.0,0.0,0.0,1.0
4,0,1,-0.651963,1,0,0.665394,76.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
1979,1,1,1.702415,1,1,-1.178997,50.0,0.0,0.0,1.0
1980,1,1,1.702415,0,0,-1.178997,55.0,0.0,0.0,1.0
1981,1,1,1.702415,0,1,-1.252773,46.0,0.0,0.0,1.0
1982,1,1,1.702415,1,1,-0.662568,53.0,0.0,0.0,1.0


4. Генерация отчёта

In [22]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Data Profiling Report", explorative=True)
profile.to_file(config['reports']['ydata_report_path'])

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 39.26it/s]
c:\Users\User\anaconda3\Lib\site-packages\ydata_profiling\model\pandas\discretize_pandas.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 9 9 9]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  discretized_df.loc[:, column] = self._discretize_column(
c:\Users\User\anaconda3\Lib\site-packages\ydata_profiling\model\pandas\discretize_pandas.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[9 9 9 ... 9 9 9]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  discretized_df.loc[:, column] = self._discretize_column(
c:\Users\User\anaconda3\Lib\site-packages\ydata_profiling\model\pandas\discretize_pandas.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future erro

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]